# Swarm-style helicorder (Bokeh)

This notebook demonstrates `SwarmHelicorderBk` for strip-style dayplots with optional clipboard linking.

> Note: these examples disable Python focus callbacks (`enable_focus_interactions=False`) to keep notebook/standalone HTML output warning-free. For live Python callbacks, run as a Bokeh server app.

**Overlay API parity with MPL `Helicorder`:** `plot_tags`, `highlight`, `plot_catalog` / `plot_events`, `info`, `set_tticks`, and `set_tzticklabel`. Each overlay glyph registers its own `HoverTool` (Kind / UTC / Details for markers; span metadata for highlight quads).

**Remaining deltas vs MPL:**
- Timezone footer labels are rendered as y-tick/footer text (not mirrored MPL left/right axis styling).
- Clipboard sync uses a centered focus window policy around `focus_time`.
- Catalog rendering matches ObsPy events (origins + picks) rather than the MPL `catalog2txyzm` scatter helper.
- Raster output is via standalone HTML `save` or browser capture rather than `savefig`.
- In standalone HTML exports, Python event callbacks are not active (focus sync is interactive in server contexts).


In [1]:
from pathlib import Path

from bokeh.io import output_notebook, reset_output
from obspy import UTCDateTime, read
from obspy.core.event import Catalog, Event, Origin, Pick, WaveformStreamID

from vdapseisutils.core.swarmmpl.bokeh import SwarmHelicorderBk

# Ensure Bokeh renders inline in notebook environments.
reset_output()
output_notebook(hide_banner=True)

# Resolve the repository root no matter whether notebook runs from repo, gallery, or gallery/SwarmMPL.
_CWD = Path.cwd().resolve()
REPO = next(
    (p for p in (_CWD, *_CWD.parents) if (p / "gallery").exists()),
    _CWD,
)
EXAMPLE_DIR = REPO / "gallery/output/Helicorders"


def disable_navigation(heli):
    """Disable pan/zoom interactions and hide the Bokeh toolbar."""
    fig = heli.figure
    fig.toolbar.tools = []
    fig.toolbar_location = None
    fig.toolbar.logo = None
    fig.toolbar.active_drag = None
    fig.toolbar.active_scroll = None
    fig.toolbar.active_tap = None
    fig.toolbar.active_multi = None
    fig.toolbar.active_inspect = []
    return heli


## MPL parity: Goma (`Helicorder_tutorial.ipynb`)

This pair of cells reproduces the matplotlib tutorial block on the Goma stream: `plot_tags`, vertical markers, mixed time formats, and three `highlight` spans. Hover markers and shaded segments for UTC metadata.

> Run the **next code cell** after the imports cell (`setup`). Data file: `gallery/output/Helicorders/goma_waveform.mseed`.


In [ ]:
path = EXAMPLE_DIR / "goma_waveform.mseed"
st_goma = read(path)

print("Creating Bokeh Helicorder... (60 minute interval)")
h_goma = SwarmHelicorderBk(
    st_goma,
    interval=60,
    color="swarm",
    title="Earthquakes near Goma (" + st_goma[0].id + ")",
    enable_focus_interactions=False,
)
h_goma.plot_tags(UTCDateTime("2021/05/20 01:36:30"), color="yellow", markersize=8)
h_goma.plot_tags(
    UTCDateTime("2021/05/20 17:19:35"),
    marker="|",
    markeredgecolor="blue",
    markersize=15,
)
h_goma.plot_tags(
    UTCDateTime("2021/05/20 17:25:00"),
    marker="|",
    markeredgecolor="red",
    markersize=15,
)
h_goma.plot_tags(
    UTCDateTime("2021/05/20 17:44:45"),
    marker="|",
    markeredgecolor="black",
    markersize=15,
)
h_goma.plot_tags(
    [UTCDateTime("2021/05/20 08:31:00"), "2021/05/20 02:08:00"],
    color="yellow",
    marker="*",
    markersize=15,
)
h_goma.highlight([(UTCDateTime("2021/05/20 01:36"), UTCDateTime("2021/05/20 02:40"))])
h_goma.highlight(
    [(UTCDateTime("2021/05/20 08:30"), UTCDateTime("2021/05/20 08:35"))],
    color="black",
)
h_goma.highlight(
    [(UTCDateTime("2021/05/20 11:30"), UTCDateTime("2021/05/20 13:35"))],
    color="red",
)

out_goma = REPO / "gallery/SwarmMPL/helicorder_tutorial_bokeh_goma.html"
h_goma.save(out_goma, title="Goma Bokeh helicorder (MPL tutorial parity)")
disable_navigation(h_goma)
h_goma.show()
out_goma


### Example 1: Basic dayplot strips

Create a simple Bokeh helicorder and display it inline.


In [2]:
path = EXAMPLE_DIR / "gareloi_waveform.mseed"
st = read(path)

heli = SwarmHelicorderBk(
    st,
    interval=60,
    color="swarm",
    title="Gareloi Bokeh helicorder",
    enable_focus_interactions=False,
)
disable_navigation(heli)
heli.show()


### Example 2: Interval + styling + tick labels

Customize strip interval, color scheme, and timezone/tick labeling.


In [3]:
path = EXAMPLE_DIR / "gareloi_waveform.mseed"
st = read(path)

heli2 = SwarmHelicorderBk(
    st,
    interval=30,
    color="earthworm",
    title="Bokeh helicorder: 30 min strips",
    utc_offset_left="UTC",
    utc_offset_right="UTC+00",
    enable_focus_interactions=False,
)
heli2.set_tticks(label_spacing=2)
heli2.set_tzticklabel(custom="UTC+00", axes="left")
disable_navigation(heli2)
heli2.show()


### Example 3: Tags/catalog markers + HTML export

Add tag/catalog markers, optionally attach a clipboard view, and write standalone HTML.


In [4]:
path = EXAMPLE_DIR / "goma_waveform.mseed"
st = read(path)
st.sort()
st.trim(st[0].stats.starttime, st[0].stats.endtime)

heli3 = SwarmHelicorderBk(
    st,
    interval=15,
    color="obspy",
    title="Goma markers",
    enable_focus_interactions=False,
)
t0 = st[0].stats.starttime + 2
t1 = st[0].stats.starttime + 5
heli3.plot_tags([t0, t1], marker="diamond", color="red", markersize=12)

event = Event(
    origins=[Origin(time=t0 + 30)],
    picks=[
        Pick(
            time=t0 + 45,
            phase_hint="P",
            waveform_id=WaveformStreamID(
                network_code=st[0].stats.network,
                station_code=st[0].stats.station,
                location_code=st[0].stats.location,
                channel_code=st[0].stats.channel,
            ),
        )
    ],
)
heli3.plot_catalog(Catalog(events=[event]), plot_picks=True, plot_origins=True)

# Optional: attach a linked clipboard view. Focus-sync window is centered on focus_time.
heli3.attach_clipboard(mode="wg", window_s=600, sync_focus=True)
disable_navigation(heli3)

out = REPO / "gallery/SwarmMPL/helicorder_tutorial_bokeh_example1.html"
heli3.save(out, title="SwarmHelicorderBk Example")
out


PosixPath('/home/jwellik/PYTHON/PKG/vdapseisutils/gallery/SwarmMPL/helicorder_tutorial_bokeh_example1.html')